# 6.37 - CertCF Alpha-Rule Epsilon Ablation

Compact COMPAS notebook to compare two CertCF build strategies: an expensive `shrink + binary search` reference and a cheap `shrink + fitted alpha rule` approximation. We use model-predicted labels everywhere.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from certcf import NearestOppositeClassClearanceStrategy
from counterfactuals.datasets.loaders import CompasDataset
from counterfactuals.methods.certcf import CertCF
from scripts.benchmark import _build_torch_model_from_checkpoint

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi': 120})

DATASET = 'compas'
SEED = 42
N_SUPPORT = 500
N_QUERIES = 100
NORM = 1
CLASSIFICATION_MARGIN = 1.0e-4
BINARY_SEARCH_STEPS = 10
DEVICE = 'auto'
CHECKPOINT = ROOT / 'checkpoints/compas_classifier/best.ckpt'

rng = np.random.default_rng(SEED)
print({'dataset': DATASET, 'n_support': N_SUPPORT, 'n_queries': N_QUERIES, 'binary_search_steps': BINARY_SEARCH_STEPS})


## Load COMPAS And Sample Support Points

In [ ]:
def stratified_indices(labels: np.ndarray, n_total: int, rng: np.random.Generator) -> np.ndarray:
    labels = np.asarray(labels, dtype=np.int64)
    classes = np.unique(labels)
    base = int(n_total) // len(classes)
    remainder = int(n_total) - base * len(classes)
    chosen = []
    for pos, cls in enumerate(classes):
        idx = np.flatnonzero(labels == cls)
        take = min(len(idx), base + int(pos < remainder))
        chosen.append(rng.choice(idx, size=take, replace=False))
    out = np.concatenate(chosen)
    rng.shuffle(out)
    return out


dataset = CompasDataset(data_dir=str(ROOT / 'data'), seed=SEED)
dataset.load()
x_train_full, y_train = dataset.get_train()
x_test, y_test = dataset.get_test()
spec = dataset.spec

model = _build_torch_model_from_checkpoint(
    checkpoint=str(CHECKPOINT),
    device=DEVICE,
    dataset_module=DATASET,
    hidden_dims=[64, 32],
    dropout=0.2,
)

y_train_pred = model.predict(x_train_full).astype(np.int64)
support_idx = stratified_indices(y_train_pred, N_SUPPORT, rng)
x_support = x_train_full[support_idx]
y_support = y_train_pred[support_idx]

y_test_pred = model.predict(x_test).astype(np.int64)
query_idx = rng.choice(np.arange(len(x_test)), size=min(N_QUERIES, len(x_test)), replace=False)
x_query = x_test[query_idx]
target_query = 1 - y_test_pred[query_idx]

summary = pd.DataFrame([{
    'dataset': DATASET,
    'encoded_dim': x_train_full.shape[1],
    'train_rows': len(x_train_full),
    'support_rows': len(x_support),
    'query_rows': len(x_query),
    'train_label_agreement': float(np.mean(y_train_pred == y_train)),
}])
display(summary.round(4))
print('support counts by model prediction:', dict(zip(*np.unique(y_support, return_counts=True))))


## Build Expensive Reference: Shrink + Binary Search

We start from the nearest-opposite-class clearance itself (`alpha=1`). Adaptive shrinkage finds a center-certified radius, and binary refinement expands it inside the bracket between the first certified radius and the previous failed radius.


In [ ]:
def collect_epsilon_diagnostics(method: CertCF) -> pd.DataFrame:
    rows = []
    atlas = method.atlas
    for label in atlas.class_labels:
        bd = atlas.bounds[int(label)]
        eps_initial = np.asarray(bd['eps_initial'], dtype=float)
        eps_final = np.asarray(bd['eps'], dtype=float)
        certified = np.asarray(bd['adaptive_eps_center_certified'], dtype=bool)
        shrinks = np.asarray(bd['adaptive_eps_n_shrinks'], dtype=float)
        binary_steps = np.asarray(bd['adaptive_eps_n_binary_steps'], dtype=float)
        slack = np.asarray(bd['adaptive_eps_center_slack'], dtype=float)
        alpha_opt = np.divide(eps_final, eps_initial, out=np.full_like(eps_final, np.nan), where=eps_initial > 0)
        rows.append(pd.DataFrame({
            'class_label': int(label),
            'clearance': eps_initial,
            'eps_opt': eps_final,
            'alpha_opt': alpha_opt,
            'center_certified': certified,
            'n_shrinks': shrinks,
            'n_binary_steps': binary_steps,
            'center_slack': slack,
        }))
    return pd.concat(rows, ignore_index=True)


binary_method = CertCF(
    model=model,
    norm=NORM,
    distance_norm=NORM,
    lirpa_method='backward',
    eps_strategy=NearestOppositeClassClearanceStrategy(alpha=1.0),
    batch_size=128,
    ohe_slices=list(spec.categorical_slices),
    default_query_method='nearest_anchor',
    query_k_candidates=5,
    k_per_class=None,
    classification_margin=CLASSIFICATION_MARGIN,
    random_seed=SEED,
    adaptive_eps=True,
    adaptive_eps_shrink_factor=0.5,
    adaptive_eps_max_shrinks=8,
    adaptive_eps_min=1.0e-6,
    adaptive_eps_center_tol=1.0e-6,
    adaptive_eps_binary_search_steps=BINARY_SEARCH_STEPS,
)

t0 = time.perf_counter()
binary_method.fit(x_train=x_support, y_train=y_support)
build_time_s = time.perf_counter() - t0

EPSILON_DF = collect_epsilon_diagnostics(binary_method)
BINARY_BUILD_SUMMARY = pd.DataFrame([{
    'build_time_s': build_time_s,
    'n_polytopes': len(EPSILON_DF),
    'center_certified_pct': 100.0 * EPSILON_DF['center_certified'].mean(),
    'mean_lirpa_calls': float((1 + EPSILON_DF['n_shrinks'] + EPSILON_DF['n_binary_steps']).mean()),
    'median_clearance': float(EPSILON_DF['clearance'].median()),
    'median_eps_opt': float(EPSILON_DF['eps_opt'].median()),
}])
display(BINARY_BUILD_SUMMARY.round(4))


## Fit The Alpha Rule

In [ ]:
FIT_DF = EPSILON_DF[
    EPSILON_DF['center_certified']
    & np.isfinite(EPSILON_DF['clearance'])
    & np.isfinite(EPSILON_DF['eps_opt'])
    & (EPSILON_DF['clearance'] > 0)
].copy()

x = FIT_DF['clearance'].to_numpy(dtype=float)
y = FIT_DF['eps_opt'].to_numpy(dtype=float)
alpha_hat = float(np.dot(x, y) / np.dot(x, x))
y_hat = alpha_hat * x
residual = y - y_hat
r2 = 1.0 - float(np.sum(residual ** 2) / np.sum((y - y.mean()) ** 2))
mae = float(np.mean(np.abs(residual)))
rmse = float(np.sqrt(np.mean(residual ** 2)))
q25, q75 = np.quantile(FIT_DF['alpha_opt'], [0.25, 0.75])

METRICS_DF = pd.DataFrame([{
    'alpha_hat': alpha_hat,
    'r2': r2,
    'mae': mae,
    'rmse': rmse,
    'alpha_opt_median': float(FIT_DF['alpha_opt'].median()),
    'alpha_opt_iqr': float(q75 - q25),
    'center_certified_pct': 100.0 * float(EPSILON_DF['center_certified'].mean()),
    'fit_points': len(FIT_DF),
}])
display(METRICS_DF.round(4))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))

axes[0].scatter(FIT_DF['clearance'], FIT_DF['eps_opt'], s=12, alpha=0.55)
limit = float(max(FIT_DF['clearance'].max(), FIT_DF['eps_opt'].max()))
grid = np.linspace(0, limit, 200)
axes[0].plot(grid, grid, color='0.35', linestyle='--', linewidth=1.2, label='clearance ceiling')
axes[0].plot(grid, alpha_hat * grid, color='#0072B2', linewidth=2.0, label=fr'fitted $\alpha={alpha_hat:.3f}$')
axes[0].set_xlabel('nearest opposite-class L1 clearance')
axes[0].set_ylabel('binary-refined center-certified epsilon')
axes[0].set_title('Optimal epsilon proxy vs alpha rule')
axes[0].legend(frameon=False)
axes[0].grid(True, alpha=0.25)

axes[1].hist(FIT_DF['alpha_opt'], bins=30, density=True, color='#009E73', alpha=0.75)
axes[1].axvline(alpha_hat, color='#D55E00', linewidth=2.0, label=fr'fitted $\alpha={alpha_hat:.3f}$')
axes[1].set_xlabel(r'$\epsilon_i^\star / d_i$')
axes[1].set_ylabel('density')
axes[1].set_title('Distribution of per-polytope alpha')
axes[1].legend(frameon=False)
axes[1].grid(True, alpha=0.25)

fig.tight_layout()
plt.show()


## Compare Against Shrink + Fitted Alpha Rule

The fitted alpha rule uses the same support/query data and the same adaptive shrink mechanism, but it disables binary refinement. This tests whether a single alpha multiplier is a good cheap approximation of the expensive per-polytope search.


In [ ]:
def build_alpha_rule_method(alpha: float) -> CertCF:
    return CertCF(
        model=model,
        norm=NORM,
        distance_norm=NORM,
        lirpa_method='backward',
        eps_strategy=NearestOppositeClassClearanceStrategy(alpha=float(alpha)),
        batch_size=128,
        ohe_slices=list(spec.categorical_slices),
        default_query_method='nearest_anchor',
        query_k_candidates=5,
        k_per_class=None,
        classification_margin=CLASSIFICATION_MARGIN,
        random_seed=SEED,
        adaptive_eps=True,
        adaptive_eps_shrink_factor=0.5,
        adaptive_eps_max_shrinks=8,
        adaptive_eps_min=1.0e-6,
        adaptive_eps_center_tol=1.0e-6,
        adaptive_eps_binary_search_steps=0,
    )


def method_query_summary(method: CertCF, label: str) -> dict:
    t0 = time.perf_counter()
    results = method.generate_batch(x_query, target_class=target_query)
    query_time_s = (time.perf_counter() - t0) / max(1, len(results))
    success = np.array([bool(r.success and r.x_cf is not None) for r in results], dtype=bool)
    l1 = np.array([
        np.linalg.norm(r.x_cf - x_query[i], ord=1)
        for i, r in enumerate(results)
        if success[i]
    ], dtype=float)
    return {
        'variant': label,
        'validity_pct': 100.0 * float(success.mean()),
        'l1_mean': float(l1.mean()) if len(l1) else np.nan,
        'l1_median': float(np.median(l1)) if len(l1) else np.nan,
        'query_time_s': float(query_time_s),
    }


def build_summary(poly_df: pd.DataFrame, label: str, build_time_s: float) -> dict:
    return {
        'variant': label,
        'build_time_s': float(build_time_s),
        'n_polytopes': len(poly_df),
        'center_certified_pct': 100.0 * float(poly_df['center_certified'].mean()),
        'mean_lirpa_calls': float((1 + poly_df['n_shrinks'] + poly_df['n_binary_steps']).mean()),
        'median_eps_final': float(poly_df['eps_opt'].median()),
        'median_alpha_final': float(poly_df['alpha_opt'].median()),
    }


alpha_rule_method = build_alpha_rule_method(alpha_hat)
t0 = time.perf_counter()
alpha_rule_method.fit(x_train=x_support, y_train=y_support)
alpha_rule_build_time_s = time.perf_counter() - t0

ALPHA_RULE_EPSILON_DF = collect_epsilon_diagnostics(alpha_rule_method)

COMPARISON_BUILD_DF = pd.DataFrame([
    build_summary(EPSILON_DF, f'shrink + binary search ({BINARY_SEARCH_STEPS} steps)', build_time_s),
    build_summary(ALPHA_RULE_EPSILON_DF, fr'shrink + alpha rule ($\alpha={alpha_hat:.3f}$)', alpha_rule_build_time_s),
])
display(COMPARISON_BUILD_DF.round(4))

COMPARISON_QUERY_DF = pd.DataFrame([
    method_query_summary(binary_method, f'shrink + binary search ({BINARY_SEARCH_STEPS} steps)'),
    method_query_summary(alpha_rule_method, fr'shrink + alpha rule ($\alpha={alpha_hat:.3f}$)'),
])
display(COMPARISON_QUERY_DF.round(4))

def epsilon_distribution_summary(df: pd.DataFrame, label: str) -> dict:
    eps = df['eps_opt'].to_numpy(dtype=float)
    ratio = df['alpha_opt'].to_numpy(dtype=float)
    return {
        'variant': label,
        'eps_mean': float(np.mean(eps)),
        'eps_median': float(np.median(eps)),
        'eps_q25': float(np.quantile(eps, 0.25)),
        'eps_q75': float(np.quantile(eps, 0.75)),
        'eps_q90': float(np.quantile(eps, 0.90)),
        'alpha_ratio_median': float(np.median(ratio)),
    }


EPSILON_DISTRIBUTION_DF = pd.DataFrame([
    epsilon_distribution_summary(EPSILON_DF, f'shrink + binary search ({BINARY_SEARCH_STEPS} steps)'),
    epsilon_distribution_summary(ALPHA_RULE_EPSILON_DF, fr'shrink + alpha rule ($\alpha={alpha_hat:.3f}$)'),
])
display(EPSILON_DISTRIBUTION_DF.round(4))


## Visualize Epsilon Distributions


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))

axes[0].hist(EPSILON_DF['eps_opt'], bins=30, density=True, alpha=0.45, label='binary search')
axes[0].hist(ALPHA_RULE_EPSILON_DF['eps_opt'], bins=30, density=True, alpha=0.45, label='alpha rule')
axes[0].set_xlabel(r'final $\epsilon_i$')
axes[0].set_ylabel('density')
axes[0].set_title('Distribution of final epsilon')
axes[0].legend(frameon=False)
axes[0].grid(True, alpha=0.25)

axes[1].hist(EPSILON_DF['alpha_opt'], bins=30, density=True, alpha=0.45, label='binary search')
axes[1].hist(ALPHA_RULE_EPSILON_DF['alpha_opt'], bins=30, density=True, alpha=0.45, label='alpha rule')
axes[1].axvline(alpha_hat, color='#D55E00', linewidth=2.0, label=fr'fitted $\alpha={alpha_hat:.3f}$')
axes[1].set_xlabel(r'final $\epsilon_i / d_i$')
axes[1].set_ylabel('density')
axes[1].set_title('Distribution of final radius ratio')
axes[1].legend(frameon=False)
axes[1].grid(True, alpha=0.25)

fig.tight_layout()
plt.show()


## Estimate Polytope Volumes

We estimate how much of each final L1 ball is retained by the LiRPA halfspace constraints. Exact polytope volume is expensive; here we use uniform Monte Carlo samples inside the final L1 ball. The estimated certified volume is the exact L1-ball volume times the retained fraction.

In [ ]:
import math

N_VOLUME_SAMPLES = 2048
VOLUME_TOL = 1.0e-8
volume_rng = np.random.default_rng(SEED + 137)


def sample_uniform_l1_ball(center: np.ndarray, eps: float, n_samples: int, rng: np.random.Generator) -> np.ndarray:
    center = np.asarray(center, dtype=np.float64)
    d = center.size
    expo = rng.exponential(scale=1.0, size=(n_samples, d + 1))
    magnitudes = expo[:, :d] / expo.sum(axis=1, keepdims=True)
    signs = rng.choice(np.array([-1.0, 1.0]), size=(n_samples, d))
    return center[None, :] + float(eps) * signs * magnitudes


def log_l1_ball_volume(eps: float, dim: int) -> float:
    if eps <= 0:
        return -np.inf
    return dim * math.log(2.0) + dim * math.log(float(eps)) - math.lgamma(dim + 1)


def estimate_volume_diagnostics(method: CertCF, variant: str, n_samples: int = N_VOLUME_SAMPLES) -> pd.DataFrame:
    rows = []
    atlas = method.atlas
    for label in atlas.class_labels:
        bd = atlas.bounds[int(label)]
        centers = np.asarray(bd['X'], dtype=np.float64)
        eps_final = np.asarray(bd['eps'], dtype=np.float64)
        eps_initial = np.asarray(bd.get('eps_initial', eps_final), dtype=np.float64)
        certified = np.asarray(bd.get('adaptive_eps_center_certified', np.ones(len(centers), dtype=bool)), dtype=bool)
        lA = np.asarray(bd['lA'], dtype=np.float64)
        lbias = np.asarray(bd['lbias'], dtype=np.float64)
        dim = centers.shape[1]
        for idx, center in enumerate(centers):
            eps = float(eps_final[idx])
            samples = sample_uniform_l1_ball(center, eps, n_samples, volume_rng)
            A = lA[idx].reshape(-1, dim)
            b = lbias[idx].reshape(-1)
            if len(b):
                slack = samples @ A.T + b[None, :] - CLASSIFICATION_MARGIN
                inside = np.all(slack >= -VOLUME_TOL, axis=1)
            else:
                inside = np.ones(n_samples, dtype=bool)
            retained = float(inside.mean())
            retained_for_log = max(retained, 0.5 / n_samples)
            log_ball = log_l1_ball_volume(eps, dim)
            rows.append({
                'variant': variant,
                'class_label': int(label),
                'polytope_idx': int(idx),
                'dim': int(dim),
                'eps_initial': float(eps_initial[idx]),
                'eps_final': eps,
                'center_certified': bool(certified[idx]),
                'retained_fraction': retained,
                'zero_hit': bool(retained == 0.0),
                'log_l1_ball_volume': log_ball,
                'log_estimated_polytope_volume': log_ball + math.log(retained_for_log),
            })
    return pd.DataFrame(rows)


BINARY_VOLUME_DF = estimate_volume_diagnostics(
    binary_method,
    f'shrink + binary search ({BINARY_SEARCH_STEPS} steps)',
)
ALPHA_RULE_VOLUME_DF = estimate_volume_diagnostics(
    alpha_rule_method,
    fr'shrink + alpha rule ($\alpha={alpha_hat:.3f}$)',
)
VOLUME_DF = pd.concat([BINARY_VOLUME_DF, ALPHA_RULE_VOLUME_DF], ignore_index=True)

VOLUME_SUMMARY_DF = (
    VOLUME_DF
    .groupby('variant', observed=True)
    .agg(
        n_polytopes=('retained_fraction', 'size'),
        retained_mean=('retained_fraction', 'mean'),
        retained_median=('retained_fraction', 'median'),
        retained_q25=('retained_fraction', lambda s: float(np.quantile(s, 0.25))),
        retained_q75=('retained_fraction', lambda s: float(np.quantile(s, 0.75))),
        zero_hit_pct=('zero_hit', lambda s: 100.0 * float(np.mean(s))),
        log_ball_volume_median=('log_l1_ball_volume', 'median'),
        log_polytope_volume_median=('log_estimated_polytope_volume', 'median'),
    )
    .reset_index()
)
display(VOLUME_SUMMARY_DF.round(4))


## Visualize Estimated Volumes


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.8))
variants = VOLUME_DF['variant'].drop_duplicates().tolist()
colors = ['#0072B2', '#D55E00']

for variant, color in zip(variants, colors):
    sub = VOLUME_DF[VOLUME_DF['variant'].eq(variant)]
    axes[0].hist(sub['retained_fraction'], bins=30, density=True, alpha=0.42, color=color, label=variant)
    axes[1].hist(sub['log_l1_ball_volume'], bins=30, density=True, alpha=0.42, color=color, label=variant)
    axes[2].hist(sub['log_estimated_polytope_volume'], bins=30, density=True, alpha=0.42, color=color, label=variant)

axes[0].set_xlabel('retained fraction inside final L1 ball')
axes[0].set_ylabel('density')
axes[0].set_title('LiRPA retained fraction')

axes[1].set_xlabel('log L1-ball volume')
axes[1].set_title('Raw final ball volume')

axes[2].set_xlabel('log estimated certified volume')
axes[2].set_title('Estimated polytope volume')

for ax in axes:
    ax.grid(True, alpha=0.25)
axes[0].legend(frameon=False, fontsize=8)

fig.tight_layout()
plt.show()
